In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from torch.nn.utils import parameters_to_vector, vector_to_parameters
import numpy as np
import matplotlib.pyplot as plt


In [ ]:

torch.set_num_threads(1)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        nn.init.constant_(m.bias, 0)

class PolicyNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64):
        super(PolicyNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )
        self.apply(init_weights)

    def forward(self, state):
        logits = self.net(state)
        return Categorical(logits=logits)

class ValueNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super(ValueNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.apply(init_weights)

    def forward(self, state):
        return self.net(state).squeeze(-1)

class TRPOAgent:
    def __init__(self, env, gamma=0.99, lam=0.97, max_kl=1e-2, damping=1e-2):
        obs_dim = env.observation_space.shape[0]
        action_dim = env.action_space.n
        self.policy = PolicyNetwork(obs_dim, action_dim)
        self.value_fn = ValueNetwork(obs_dim)
        self.value_optimizer = optim.Adam(self.value_fn.parameters(), lr=1e-3)

        self.gamma = gamma
        self.lam = lam
        self.max_kl = max_kl
        self.damping = damping

    def select_action(self, state):
        dist = self.policy(state)
        action = dist.sample()
        return action.item(), dist.log_prob(action), dist

    def compute_gae(self, rewards, masks, values):
        returns = []
        gae = 0
        with torch.no_grad():
            values = np.append(values, 0)
            for step in reversed(range(len(rewards))):
                delta = rewards[step] + self.gamma * values[step+1] * masks[step] - values[step]
                gae = delta + self.gamma * self.lam * masks[step] * gae
                returns.insert(0, gae + values[step])
        advs = np.array(returns) - values[:-1]
        return np.array(returns), (advs - advs.mean()) / (advs.std() + 1e-8)

    def conjugate_grad(self, fisher_vec_prod, b, iters=10, tol=1e-10):
        x = torch.zeros_like(b)
        r = b.clone()
        p = b.clone()
        rdotr = torch.dot(r, r)
        for _ in range(iters):
            Ap = fisher_vec_prod(p)
            alpha = rdotr / (torch.dot(p, Ap) + 1e-8)
            x += alpha * p
            r -= alpha * Ap
            new_rdotr = torch.dot(r, r)
            if new_rdotr < tol:
                break
            beta = new_rdotr / rdotr
            p = r + beta * p
            rdotr = new_rdotr
        return x

    def fisher_vector_product(self, states, vec):
        dists = self.policy(states)
        with torch.no_grad():
            old_dists = Categorical(logits=self.policy.net(states).detach())

        kl = torch.distributions.kl_divergence(old_dists, dists).mean()
        grads = torch.autograd.grad(kl, self.policy.parameters(), create_graph=True)
        flat_grad_kl = parameters_to_vector(grads)
        kl_v = (flat_grad_kl * vec).sum()
        grad2 = torch.autograd.grad(kl_v, self.policy.parameters())
        flat_grad2 = parameters_to_vector(grad2)
        return flat_grad2 + self.damping * vec

    def surrogate_loss(self, states, actions, old_log_probs, advantages):
        dists = self.policy(states)
        log_probs = dists.log_prob(actions)
        ratio = torch.exp(log_probs - old_log_probs)
        return -(ratio * advantages).mean()

    def linesearch(self, states, actions, old_log_probs, advantages, full_step, expected_improve_rate):
        old_params = parameters_to_vector(self.policy.parameters())
        for stepfrac in [0.5 ** i for i in range(10)]:
            new_params = old_params + stepfrac * full_step
            vector_to_parameters(new_params, self.policy.parameters())
            loss = self.surrogate_loss(states, actions, old_log_probs, advantages)
            kl = torch.distributions.kl_divergence(
                Categorical(logits=self.policy.net(states).detach()),
                self.policy(states)
            ).mean()
            actual_improve = loss.item() - self.surrogate_loss(states, actions, old_log_probs, advantages).item()
            expected_improve = expected_improve_rate * stepfrac
            if actual_improve < 0 and kl < self.max_kl:
                return True, new_params
        vector_to_parameters(old_params, self.policy.parameters())
        return False, old_params

    def update(self, batch):
        states = torch.tensor(np.vstack(batch['states']), dtype=torch.float32)
        actions = torch.tensor(batch['actions'], dtype=torch.int64)
        old_log_probs = torch.tensor(batch['log_probs'], dtype=torch.float32)
        returns = torch.tensor(batch['returns'], dtype=torch.float32)
        advantages = torch.tensor(batch['advantages'], dtype=torch.float32)
        loss = self.surrogate_loss(states, actions, old_log_probs, advantages)
        grads = torch.autograd.grad(loss, self.policy.parameters())
        loss_grad = parameters_to_vector(grads).detach()
        step_dir = self.conjugate_grad(lambda v: self.fisher_vector_product(states, v), -loss_grad)
        shs = 0.5 * (step_dir * self.fisher_vector_product(states, step_dir)).sum(0)
        lm = torch.sqrt(shs / self.max_kl)
        full_step = step_dir / lm
        expected_improve = -(loss_grad * full_step).sum(0)
        success, new_params = self.linesearch(states, actions, old_log_probs, advantages, full_step, expected_improve)
        vector_to_parameters(new_params, self.policy.parameters())
        for _ in range(80):
            values = self.value_fn(states)
            value_loss = nn.MSELoss()(values, returns)
            self.value_optimizer.zero_grad()
            value_loss.backward()
            self.value_optimizer.step()


In [ ]:
def train():
    env = gym.make('CartPole-v1')
    agent = TRPOAgent(env)
    max_iter = 1000
    batch_size = 5000
    window_size = 50  # rolling window for averaging
    total_rewards = []

    for iteration in range(max_iter):
        batch = {'states': [], 'actions': [], 'log_probs': [], 'masks': [], 'returns': [], 'advantages': []}
        timesteps = 0
        ep_rewards = []
        while timesteps < batch_size:
            state, _ = env.reset()
            done = False
            ep_states, ep_actions, ep_rs, ep_log_probs, ep_masks = [], [], [], [], []
            while not done and timesteps < batch_size:
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                action, log_prob, _ = agent.select_action(state_tensor)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                ep_states.append(state)
                ep_actions.append(action)
                ep_rs.append(reward)
                ep_log_probs.append(log_prob.item())
                ep_masks.append(1 - int(done))
                state = next_state
                timesteps += 1
            returns, advantages = agent.compute_gae(
                ep_rs, ep_masks,
                agent.value_fn(torch.tensor(np.vstack(ep_states), dtype=torch.float32)).detach().numpy()
            )
            batch['states'] += ep_states
            batch['actions'] += ep_actions
            batch['log_probs'] += ep_log_probs
            batch['masks'] += ep_masks
            batch['returns'] += list(returns)
            batch['advantages'] += list(advantages)
            total_rewards.append(sum(ep_rs))
            ep_rewards.append(sum(ep_rs))

        agent.update(batch)
        avg_last = np.mean(total_rewards[-window_size:])
        print(f"Iteration {iteration}: Average Return (last {window_size} eps): {avg_last:.2f}")

    # Plot results
    rolling_avg = [np.mean(total_rewards[max(0, i-window_size+1):(i+1)]) for i in range(len(total_rewards))]
    plt.plot(total_rewards, label='Total Reward per Episode')
    plt.plot(rolling_avg, label=f'{window_size}-Episode Rolling Average')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('TRPO - CartPole')
    plt.legend()
    plt.show()


train()
